In [50]:
# ============================================================
# STAGE 1 — DOCUMENT CHARACTERISATION AND QUALITY ASSESSMENT
# D13 — Unemployment Rates by Country of Birth
# ============================================================

from google.colab import files
from pathlib import Path
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter

import hashlib
import json
import platform
import re
import sys

import pandas as pd

In [51]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D13"

DOCUMENT_NAME = (
    "Eurostat — Unemployment rates by country of birth"
)

SOURCE_FORMAT = "XLSX"

SUMMARY_SHEET = "Summary"

SELECTED_SHEETS = [
    "Sheet 1",
    "Sheet 2",
    "Sheet 3",
    "Sheet 4",
    "Sheet 5"
]

SELECTED_GEOGRAPHIES = [
    "European Union - 27 countries (from 2020)",
    "Belgium",
    "Germany",
    "Spain",
    "Portugal"
]

SELECTED_YEARS = [
    2020,
    2022,
    2024
]

EXPECTED_REFERENCE_RECORD_COUNT = (
    len(SELECTED_SHEETS)
    * len(SELECTED_GEOGRAPHIES)
    * len(SELECTED_YEARS)
)

REFERENCE_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Reporting Period",
    "Source Location"
]

REFERENCE_CATEGORY = (
    "Labour market time series"
)

REFERENCE_TOPIC = (
    "Unemployment rate by country of birth"
)

OUTPUT_DIR = Path(
    "outputs_D13_stage1"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DOCUMENT_CHARACTERISATION_PATH = (
    OUTPUT_DIR
    / "D13_document_characterisation.json"
)

WORKBOOK_INTEGRITY_PATH = (
    OUTPUT_DIR
    / "D13_workbook_integrity.json"
)

SHEET_CHARACTERISATION_PATH = (
    OUTPUT_DIR
    / "D13_sheet_characterisation.csv"
)

SELECTION_MATRIX_PATH = (
    OUTPUT_DIR
    / "D13_selection_matrix.csv"
)

REFERENCE_VALUES_PATH = (
    OUTPUT_DIR
    / "D13_reference_values.csv"
)

REFERENCE_METADATA_PATH = (
    OUTPUT_DIR
    / "D13_reference_metadata.json"
)

INDICATOR_ASSESSMENT_PATH = (
    OUTPUT_DIR
    / "D13_indicator_assessment.csv"
)

DIMENSION_ASSESSMENT_PATH = (
    OUTPUT_DIR
    / "D13_dimension_assessment.csv"
)

QUALITY_EVIDENCE_PATH = (
    OUTPUT_DIR
    / "D13_quality_evidence.json"
)

EXTRACTION_SCHEMA_PATH = (
    OUTPUT_DIR
    / "D13_extraction_schema.json"
)

EXTRACTION_TASK_PATH = (
    OUTPUT_DIR
    / "D13_extraction_task.txt"
)

REFERENCE_SCHEMA_PATH = (
    OUTPUT_DIR
    / "D13_reference_schema.json"
)

REFERENCE_SUMMARY_PATH = (
    OUTPUT_DIR
    / "D13_reference_summary.json"
)

REFERENCE_INTEGRITY_PATH = (
    OUTPUT_DIR
    / "D13_reference_integrity.json"
)

REFERENCE_VALUES_JSON_PATH = (
    OUTPUT_DIR
    / "D13_reference_values.json"
)

print("Document:", DOCUMENT_ID)
print(
    "Expected reference records:",
    EXPECTED_REFERENCE_RECORD_COUNT
)
print("Output directory:", OUTPUT_DIR)


Document: D13
Expected reference records: 75
Output directory: outputs_D13_stage1


In [52]:
# ============================================================
# 2. Upload original workbook
# ============================================================

print(
    "Upload the original D13 XLSX workbook."
)

uploaded = files.upload()

xlsx_paths = [
    Path(filename)
    for filename in uploaded
    if filename.lower().endswith(".xlsx")
]

if len(xlsx_paths) != 1:
    raise ValueError(
        "Upload exactly one XLSX workbook."
    )

SOURCE_PATH = xlsx_paths[0]

print("Source file:", SOURCE_PATH.name)
print(
    "Source size:",
    f"{SOURCE_PATH.stat().st_size:,} bytes"
)


Upload the original D13 XLSX workbook.


Saving D13 - Unemployment rates by country of birth_2026.xlsx to D13 - Unemployment rates by country of birth_2026.xlsx
Source file: D13 - Unemployment rates by country of birth_2026.xlsx
Source size: 88,159 bytes


In [53]:
# ============================================================
# 3. File hashing and workbook loading
# ============================================================

def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


SOURCE_SHA256 = sha256_file(
    SOURCE_PATH
)

workbook_values = load_workbook(
    SOURCE_PATH,
    data_only=True,
    read_only=False
)

workbook_formulas = load_workbook(
    SOURCE_PATH,
    data_only=False,
    read_only=False
)

sheet_names = workbook_values.sheetnames

print("Source SHA-256:", SOURCE_SHA256)
print("Worksheet count:", len(sheet_names))
print("Worksheets:", sheet_names)


/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Source SHA-256: 13f6d031c0e5c888d5632ad20853d16e603f462cdf3b58c0dd40057824d10f9a
Worksheet count: 16
Worksheets: ['Summary', 'Sheet 1', 'Sheet 2', 'Sheet 3', 'Sheet 4', 'Sheet 5', 'Sheet 6', 'Sheet 7', 'Sheet 8', 'Sheet 9', 'Sheet 10', 'Sheet 11', 'Sheet 12', 'Sheet 13', 'Sheet 14', 'Sheet 15']


In [54]:
# ============================================================
# 4. Workbook structure validation
# ============================================================

EXPECTED_SHEETS = [
    "Summary"
] + [
    f"Sheet {number}"
    for number in range(1, 16)
]

worksheet_count_valid = (
    len(sheet_names) == 16
)

worksheet_names_valid = (
    sheet_names == EXPECTED_SHEETS
)

summary_sheet_present = (
    SUMMARY_SHEET in sheet_names
)

selected_sheets_present = all(
    sheet in sheet_names
    for sheet in SELECTED_SHEETS
)

workbook_structure_valid = all(
    [
        worksheet_count_valid,
        worksheet_names_valid,
        summary_sheet_present,
        selected_sheets_present
    ]
)

WORKBOOK_INTEGRITY = {
    "document_id": DOCUMENT_ID,
    "source_file": SOURCE_PATH.name,
    "source_file_sha256": SOURCE_SHA256,
    "expected_worksheet_count": 16,
    "observed_worksheet_count":
        len(sheet_names),
    "expected_worksheet_names":
        EXPECTED_SHEETS,
    "observed_worksheet_names":
        sheet_names,
    "worksheet_count_valid":
        worksheet_count_valid,
    "worksheet_names_valid":
        worksheet_names_valid,
    "summary_sheet_present":
        summary_sheet_present,
    "selected_sheets_present":
        selected_sheets_present,
    "workbook_structure_valid":
        workbook_structure_valid
}

print(
    json.dumps(
        WORKBOOK_INTEGRITY,
        ensure_ascii=False,
        indent=2
    )
)

if not workbook_structure_valid:
    raise AssertionError(
        "D13 workbook structure validation failed."
    )


{
  "document_id": "D13",
  "source_file": "D13 - Unemployment rates by country of birth_2026.xlsx",
  "source_file_sha256": "13f6d031c0e5c888d5632ad20853d16e603f462cdf3b58c0dd40057824d10f9a",
  "expected_worksheet_count": 16,
  "observed_worksheet_count": 16,
  "expected_worksheet_names": [
    "Summary",
    "Sheet 1",
    "Sheet 2",
    "Sheet 3",
    "Sheet 4",
    "Sheet 5",
    "Sheet 6",
    "Sheet 7",
    "Sheet 8",
    "Sheet 9",
    "Sheet 10",
    "Sheet 11",
    "Sheet 12",
    "Sheet 13",
    "Sheet 14",
    "Sheet 15"
  ],
  "observed_worksheet_names": [
    "Summary",
    "Sheet 1",
    "Sheet 2",
    "Sheet 3",
    "Sheet 4",
    "Sheet 5",
    "Sheet 6",
    "Sheet 7",
    "Sheet 8",
    "Sheet 9",
    "Sheet 10",
    "Sheet 11",
    "Sheet 12",
    "Sheet 13",
    "Sheet 14",
    "Sheet 15"
  ],
  "worksheet_count_valid": true,
  "worksheet_names_valid": true,
  "summary_sheet_present": true,
  "selected_sheets_present": true,
  "workbook_structure_valid": true
}


In [55]:
# ============================================================
# 5. Summary-sheet metadata validation
# ============================================================

summary_df = pd.read_excel(
    SOURCE_PATH,
    sheet_name=SUMMARY_SHEET,
    header=None
)

summary_text = (
    " ".join(
        summary_df
        .fillna("")
        .astype(str)
        .values
        .flatten()
    )
)

dataset_identifier_present = (
    "lfsa_urgacob" in summary_text
)

dataset_title_present = (
    "Unemployment rates by country of birth"
    in summary_text
)

annual_frequency_present = (
    "Annual" in summary_text
)

percentage_unit_present = (
    "Percentage" in summary_text
)

age_class_present = (
    "From 15 to 74 years"
    in summary_text
)

summary_metadata_valid = all(
    [
        dataset_identifier_present,
        dataset_title_present,
        annual_frequency_present,
        percentage_unit_present,
        age_class_present
    ]
)

if not summary_metadata_valid:
    raise AssertionError(
        "Required D13 summary metadata was not found."
    )

print("Summary metadata valid:", summary_metadata_valid)


Summary metadata valid: True


In [56]:
# ============================================================
# 6. Sheet-level characterisation
# ============================================================

sheet_characterisation_rows = []

for sheet_name in sheet_names:
    worksheet = workbook_values[
        sheet_name
    ]

    non_empty_cells = 0
    numeric_cells = 0
    formula_cells = 0

    formula_worksheet = workbook_formulas[
        sheet_name
    ]

    for row in worksheet.iter_rows():
        for cell in row:
            if cell.value is not None:
                non_empty_cells += 1

                if isinstance(
                    cell.value,
                    (int, float)
                ) and not isinstance(
                    cell.value,
                    bool
                ):
                    numeric_cells += 1

    for row in formula_worksheet.iter_rows():
        for cell in row:
            if (
                isinstance(cell.value, str)
                and cell.value.startswith("=")
            ):
                formula_cells += 1

    row_5_value = worksheet["C5"].value
    row_6_value = worksheet["C6"].value
    row_7_value = worksheet["C7"].value
    row_8_value = worksheet["C8"].value
    row_9_value = worksheet["C9"].value

    sheet_characterisation_rows.append(
        {
            "Sheet Name": sheet_name,
            "Rows": worksheet.max_row,
            "Columns": worksheet.max_column,
            "Non-empty Cells": non_empty_cells,
            "Numeric Cells": numeric_cells,
            "Formula Cells": formula_cells,

            "Merged Range Count":
                len(
                    worksheet.merged_cells.ranges
                ),

            "Time Frequency": row_5_value,
            "Unit of Measure": row_6_value,
            "Sex": row_7_value,
            "Age Class": row_8_value,
            "Country/Region of Birth":
                row_9_value,
            "Selected for Reference Task":
                sheet_name in SELECTED_SHEETS
        }
    )


sheet_characterisation_df = pd.DataFrame(
    sheet_characterisation_rows
)

display(sheet_characterisation_df)


,Sheet Name,Rows,Columns,Non-empty Cells,Numeric Cells,Formula Cells,Merged Range Count,Time Frequency,Unit of Measure,Sex,Age Class,Country/Region of Birth,Selected for Reference Task
0,Summary,34,15,113,0,0,1,None,None,None,None,None,False
1,Sheet 1,59,21,859,277,0,10,Annual,Percentage,Total,From 15 to 74 years,EU27 countries (from 2020) except reporting co...,True
2,Sheet 2,59,21,859,317,0,10,Annual,Percentage,Total,From 15 to 74 years,Non-EU27 countries (from 2020) nor reporting c...,True
3,Sheet 3,59,21,859,333,0,10,Annual,Percentage,Total,From 15 to 74 years,Foreign country,True
4,Sheet 4,57,21,855,365,0,10,Annual,Percentage,Total,From 15 to 74 years,Reporting country,True
5,Sheet 5,57,21,855,365,0,10,Annual,Percentage,Total,From 15 to 74 years,Total,True
6,Sheet 6,59,21,859,246,0,10,Annual,Percentage,Males,From 15 to 74 years,EU27 countries (from 2020) except reporting co...,False
7,Sheet 7,59,21,859,302,0,10,Annual,Percentage,Males,From 15 to 74 years,Non-EU27 countries (from 2020) nor reporting c...,False
8,Sheet 8,59,21,859,316,0,10,Annual,Percentage,Males,From 15 to 74 years,Foreign country,False
9,Sheet 9,58,21,857,365,0,10,Annual,Percentage,Males,From 15 to 74 years,Reporting country,False


In [57]:
# ============================================================
# 7. Selection matrix construction
# ============================================================

def clean_text(value):
    if value is None:
        return ""

    return re.sub(
        r"\s+",
        " ",
        str(value)
    ).strip()


def find_year_columns(worksheet):
    year_columns = {}

    for column_index in range(
        1,
        worksheet.max_column + 1
    ):
        value = worksheet.cell(
            row=11,
            column=column_index
        ).value

        if isinstance(value, int):
            year_columns[value] = column_index

        elif (
            isinstance(value, str)
            and re.fullmatch(
                r"20\d{2}",
                value.strip()
            )
        ):
            year_columns[
                int(value.strip())
            ] = column_index

    return year_columns


def find_geography_rows(worksheet):
    geography_rows = {}

    for row_index in range(
        13,
        worksheet.max_row + 1
    ):
        label = clean_text(
            worksheet.cell(
                row=row_index,
                column=1
            ).value
        )

        if label:
            geography_rows[label] = row_index

    return geography_rows


selection_rows = []

for sheet_name in SELECTED_SHEETS:
    worksheet = workbook_values[
        sheet_name
    ]

    year_columns = find_year_columns(
        worksheet
    )

    geography_rows = find_geography_rows(
        worksheet
    )

    sex = clean_text(
        worksheet["C7"].value
    )

    age_class = clean_text(
        worksheet["C8"].value
    )

    birth_category = clean_text(
        worksheet["C9"].value
    )

    for geography in SELECTED_GEOGRAPHIES:
        if geography not in geography_rows:
            raise AssertionError(
                f"{geography!r} was not found in {sheet_name}."
            )

        row_index = geography_rows[
            geography
        ]

        for year in SELECTED_YEARS:
            if year not in year_columns:
                raise AssertionError(
                    f"Year {year} was not found in {sheet_name}."
                )

            value_column = year_columns[
                year
            ]

            flag_column = (
                value_column + 1
            )

            value_cell = worksheet.cell(
                row=row_index,
                column=value_column
            )

            flag_cell = worksheet.cell(
                row=row_index,
                column=flag_column
            )

            value = value_cell.value
            flag = clean_text(
                flag_cell.value
            )

            selection_rows.append(
                {
                    "Sheet":
                        sheet_name,

                    "Geography":
                        geography,

                    "Sex":
                        sex,

                    "Age Class":
                        age_class,

                    "Country/Region of Birth":
                        birth_category,

                    "Year":
                        year,

                    "Value":
                        value,

                    "Statistical Flag":
                        flag,

                    "Value Cell":
                        value_cell.coordinate,

                    "Flag Cell":
                        flag_cell.coordinate
                }
            )


selection_matrix_df = pd.DataFrame(
    selection_rows
)

display(selection_matrix_df.head(20))

print(
    "Selection matrix records:",
    len(selection_matrix_df)
)


,Sheet,Geography,Sex,Age Class,Country/Region of Birth,Year,Value,Statistical Flag,Value Cell,Flag Cell
0,Sheet 1,European Union - 27 countries (from 2020),Total,From 15 to 74 years,EU27 countries (from 2020) except reporting co...,2020,8.3,,L13,M13
1,Sheet 1,European Union - 27 countries (from 2020),Total,From 15 to 74 years,EU27 countries (from 2020) except reporting co...,2022,6.8,,P13,Q13
2,Sheet 1,European Union - 27 countries (from 2020),Total,From 15 to 74 years,EU27 countries (from 2020) except reporting co...,2024,6.5,,T13,U13
3,Sheet 1,Belgium,Total,From 15 to 74 years,EU27 countries (from 2020) except reporting co...,2020,6.4,,L15,M15
4,Sheet 1,Belgium,Total,From 15 to 74 years,EU27 countries (from 2020) except reporting co...,2022,5.3,,P15,Q15
5,Sheet 1,Belgium,Total,From 15 to 74 years,EU27 countries (from 2020) except reporting co...,2024,6.6,,T15,U15
6,Sheet 1,Germany,Total,From 15 to 74 years,EU27 countries (from 2020) except reporting co...,2020,4.6,b,L19,M19
7,Sheet 1,Germany,Total,From 15 to 74 years,EU27 countries (from 2020) except reporting co...,2022,3.6,,P19,Q19
8,Sheet 1,Germany,Total,From 15 to 74 years,EU27 countries (from 2020) except reporting co...,2024,3.7,,T19,U19
9,Sheet 1,Spain,Total,From 15 to 74 years,EU27 countries (from 2020) except reporting co...,2020,19.5,,L23,M23


Selection matrix records: 75


In [58]:
# ============================================================
# 8. Selected-observation validation
# ============================================================

selection_record_count_valid = (
    len(selection_matrix_df)
    == EXPECTED_REFERENCE_RECORD_COUNT
)

selected_values_numeric = bool(
    selection_matrix_df[
        "Value"
    ].map(
        lambda value: (
            isinstance(
                value,
                (int, float)
            )
            and not isinstance(
                value,
                bool
            )
        )
    ).all()
)

selected_values_non_negative = bool(
    (
        pd.to_numeric(
            selection_matrix_df["Value"],
            errors="coerce"
        )
        >= 0
    ).all()
)

selected_values_within_percentage_range = bool(
    (
        pd.to_numeric(
            selection_matrix_df["Value"],
            errors="coerce"
        )
        <= 100
    ).all()
)

selected_dimensions_complete = bool(
    selection_matrix_df[
        [
            "Sheet",
            "Geography",
            "Sex",
            "Age Class",
            "Country/Region of Birth",
            "Year",
            "Value",
            "Value Cell"
        ]
    ].notna().all().all()
)

selected_observations_valid = all(
    [
        selection_record_count_valid,
        selected_values_numeric,
        selected_values_non_negative,
        selected_values_within_percentage_range,
        selected_dimensions_complete
    ]
)

print(
    "Selection record count valid:",
    selection_record_count_valid
)

print(
    "Selected values numeric:",
    selected_values_numeric
)

print(
    "Selected values within 0–100:",
    selected_values_within_percentage_range
)

print(
    "Selected observations valid:",
    selected_observations_valid
)

if not selected_observations_valid:
    raise AssertionError(
        "D13 selected-observation validation failed."
    )


Selection record count valid: True
Selected values numeric: True
Selected values within 0–100: True
Selected observations valid: True


In [59]:
# ============================================================
# 9. Representation diagnostics
# ============================================================

statistical_flag_counts = (
    selection_matrix_df[
        "Statistical Flag"
    ]
    .fillna("")
    .replace(
        "",
        "No flag"
    )
    .value_counts()
    .to_dict()
)

flagged_reference_records = int(
    (
        selection_matrix_df[
            "Statistical Flag"
        ]
        .fillna("")
        .astype(str)
        .str.strip()
        != ""
    ).sum()
)

unflagged_reference_records = int(
    len(selection_matrix_df)
    - flagged_reference_records
)


selected_data_sheet_profile = (
    sheet_characterisation_df[
        sheet_characterisation_df[
            "Sheet Name"
        ].isin(
            SELECTED_SHEETS
        )
    ]
)


selected_non_empty_cells = int(
    selected_data_sheet_profile[
        "Non-empty Cells"
    ].sum()
)

selected_numeric_cells = int(
    selected_data_sheet_profile[
        "Numeric Cells"
    ].sum()
)

numeric_cell_density = (
    selected_numeric_cells
    / selected_non_empty_cells
    if selected_non_empty_cells
    else 0.0
)


REPRESENTATION_DIAGNOSTICS = {
    "document_id":
        DOCUMENT_ID,

    "selected_reference_records":
        int(
            len(selection_matrix_df)
        ),

    "flagged_reference_records":
        flagged_reference_records,

    "unflagged_reference_records":
        unflagged_reference_records,

    "statistical_flag_counts":
        statistical_flag_counts,

    "selected_sheet_non_empty_cells":
        selected_non_empty_cells,

    "selected_sheet_numeric_cells":
        selected_numeric_cells,

    "selected_sheet_numeric_cell_density":
        round(
            numeric_cell_density,
            3
        ),

    "value_flag_column_pairs":
        True,

    "year_headers_merged_across_value_flag_pairs":
        True,

    "sheet_level_dimensions":
        [
            "Sex",
            "Age Class",
            "Country/Region of Birth"
        ],

    "row_level_dimension":
        "Geography",

    "column_level_dimension":
        "Reporting year"
}


print(
    json.dumps(
        REPRESENTATION_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    )
)

{
  "document_id": "D13",
  "selected_reference_records": 75,
  "flagged_reference_records": 17,
  "unflagged_reference_records": 58,
  "statistical_flag_counts": {
    "No flag": 58,
    "d": 10,
    "b": 5,
    "u": 2
  },
  "selected_sheet_non_empty_cells": 4287,
  "selected_sheet_numeric_cells": 1657,
  "selected_sheet_numeric_cell_density": 0.387,
  "value_flag_column_pairs": true,
  "year_headers_merged_across_value_flag_pairs": true,
  "sheet_level_dimensions": [
    "Sex",
    "Age Class",
    "Country/Region of Birth"
  ],
  "row_level_dimension": "Geography",
  "column_level_dimension": "Reporting year"
}


In [60]:
# ============================================================
# 10. Reference dataset construction
# ============================================================

reference_rows = []

for _, row in selection_matrix_df.iterrows():
    description_parts = [
        f"Geography: {row['Geography']}",
        f"Sex: {row['Sex']}",
        f"Age class: {row['Age Class']}",
        (
            "Country/region of birth: "
            f"{row['Country/Region of Birth']}"
        )
    ]

    statistical_flag = clean_text(
        row["Statistical Flag"]
    )

    if statistical_flag:
        description_parts.append(
            f"Statistical flag: {statistical_flag}"
        )

    reference_rows.append(
        {
            "Category":
                REFERENCE_CATEGORY,

            "Topic":
                REFERENCE_TOPIC,

            "Description":
                "; ".join(
                    description_parts
                ),

            "Value":
                float(row["Value"]),

            "Unit":
                "percent",

            "Reporting Period":
                str(int(row["Year"])),

            "Source Location":
                (
                    f"{row['Sheet']}, "
                    f"cell {row['Value Cell']}"
                )
        }
    )


reference_values_df = pd.DataFrame(
    reference_rows,
    columns=REFERENCE_FIELDS
)

display(reference_values_df.head(20))

print(
    "Reference records:",
    len(reference_values_df)
)


,Category,Topic,Description,Value,Unit,Reporting Period,Source Location
0,Labour market time series,Unemployment rate by country of birth,Geography: European Union - 27 countries (from...,8.3,percent,2020,"Sheet 1, cell L13"
1,Labour market time series,Unemployment rate by country of birth,Geography: European Union - 27 countries (from...,6.8,percent,2022,"Sheet 1, cell P13"
2,Labour market time series,Unemployment rate by country of birth,Geography: European Union - 27 countries (from...,6.5,percent,2024,"Sheet 1, cell T13"
3,Labour market time series,Unemployment rate by country of birth,Geography: Belgium; Sex: Total; Age class: Fro...,6.4,percent,2020,"Sheet 1, cell L15"
4,Labour market time series,Unemployment rate by country of birth,Geography: Belgium; Sex: Total; Age class: Fro...,5.3,percent,2022,"Sheet 1, cell P15"
5,Labour market time series,Unemployment rate by country of birth,Geography: Belgium; Sex: Total; Age class: Fro...,6.6,percent,2024,"Sheet 1, cell T15"
6,Labour market time series,Unemployment rate by country of birth,Geography: Germany; Sex: Total; Age class: Fro...,4.6,percent,2020,"Sheet 1, cell L19"
7,Labour market time series,Unemployment rate by country of birth,Geography: Germany; Sex: Total; Age class: Fro...,3.6,percent,2022,"Sheet 1, cell P19"
8,Labour market time series,Unemployment rate by country of birth,Geography: Germany; Sex: Total; Age class: Fro...,3.7,percent,2024,"Sheet 1, cell T19"
9,Labour market time series,Unemployment rate by country of birth,Geography: Spain; Sex: Total; Age class: From ...,19.5,percent,2020,"Sheet 1, cell L23"


Reference records: 75


In [61]:
# ============================================================
# 11. Reference-value validation
# ============================================================

reference_schema_valid = (
    reference_values_df.columns.tolist()
    == REFERENCE_FIELDS
)

reference_record_count_valid = (
    len(reference_values_df)
    == EXPECTED_REFERENCE_RECORD_COUNT
)

reference_null_count = int(
    reference_values_df.isna().sum().sum()
)

reference_duplicate_count = int(
    reference_values_df.duplicated().sum()
)

reference_values_numeric = bool(
    reference_values_df[
        "Value"
    ].map(
        lambda value: (
            isinstance(
                value,
                (int, float)
            )
            and not isinstance(
                value,
                bool
            )
        )
    ).all()
)

reference_periods_valid = bool(
    reference_values_df[
        "Reporting Period"
    ].str.fullmatch(
        r"20\d{2}"
    ).all()
)

reference_locations_valid = bool(
    reference_values_df[
        "Source Location"
    ].str.fullmatch(
        r"Sheet \d+, cell [A-Z]+\d+"
    ).all()
)

reference_category_valid = bool(
    (
        reference_values_df["Category"]
        == REFERENCE_CATEGORY
    ).all()
)

reference_topic_valid = bool(
    (
        reference_values_df["Topic"]
        == REFERENCE_TOPIC
    ).all()
)

reference_unit_valid = bool(
    (
        reference_values_df["Unit"]
        == "percent"
    ).all()
)

reference_integrity_valid = all(
    [
        reference_schema_valid,
        reference_record_count_valid,
        reference_null_count == 0,
        reference_duplicate_count == 0,
        reference_values_numeric,
        reference_periods_valid,
        reference_locations_valid,
        reference_category_valid,
        reference_topic_valid,
        reference_unit_valid
    ]
)

REFERENCE_VALIDATION = {
    "document_id": DOCUMENT_ID,
    "expected_reference_records":
        EXPECTED_REFERENCE_RECORD_COUNT,
    "observed_reference_records":
        int(len(reference_values_df)),
    "reference_schema_valid":
        reference_schema_valid,
    "reference_record_count_valid":
        reference_record_count_valid,
    "reference_null_count":
        reference_null_count,
    "reference_duplicate_count":
        reference_duplicate_count,
    "reference_values_numeric":
        reference_values_numeric,
    "reference_periods_valid":
        reference_periods_valid,
    "reference_locations_valid":
        reference_locations_valid,
    "reference_category_valid":
        reference_category_valid,
    "reference_topic_valid":
        reference_topic_valid,
    "reference_unit_valid":
        reference_unit_valid,
    "reference_integrity_valid":
        reference_integrity_valid
}

print(
    json.dumps(
        REFERENCE_VALIDATION,
        ensure_ascii=False,
        indent=2
    )
)

if not reference_integrity_valid:
    raise AssertionError(
        "D13 reference-value validation failed."
    )


{
  "document_id": "D13",
  "expected_reference_records": 75,
  "observed_reference_records": 75,
  "reference_schema_valid": true,
  "reference_record_count_valid": true,
  "reference_null_count": 0,
  "reference_duplicate_count": 0,
  "reference_values_numeric": true,
  "reference_periods_valid": true,
  "reference_locations_valid": true,
  "reference_category_valid": true,
  "reference_topic_valid": true,
  "reference_unit_valid": true,
  "reference_integrity_valid": true
}


In [62]:
# ============================================================
# 12. Document-level characterisation
# ============================================================

DOCUMENT_CHARACTERISATION = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "source_file":
        SOURCE_PATH.name,

    "source_file_sha256":
        SOURCE_SHA256,

    "source_format":
        SOURCE_FORMAT,

    "document_class":
        "Multi-sheet statistical workbook",

    "worksheet_count":
        len(
            sheet_names
        ),

    "summary_sheet":
        SUMMARY_SHEET,

    "data_sheet_count":
        len(sheet_names) - 1,

    "machine_readable":
        True,

    "text_extractable":
        True,

    "ocr_required":
        False,

    "time_frequency":
        "Annual",

    "unit_of_measure":
        "Percentage",

    "age_class":
        "From 15 to 74 years",

    "temporal_coverage": {
        "minimum_year":
            2015,

        "maximum_year":
            2024
    },

    "multi_sheet_structure":
        True,

    "repeated_data_sheet_template":
        True,

    "years_stored_in_columns":
        True,

    "value_flag_column_pairs":
        True,

    "merged_year_headers_present":
        bool(
            sheet_characterisation_df[
                "Merged Range Count"
            ].sum()
            > 0
        ),

    "sheet_level_dimensions":
        [
            "Sex",
            "Age Class",
            "Country/Region of Birth"
        ],

    "row_level_dimensions":
        [
            "Geography"
        ],

    "column_level_dimensions":
        [
            "Reporting year",
            "Statistical flag"
        ],

    "statistical_flags_present":
        bool(
            flagged_reference_records
            > 0
        ),

    "selected_reference_scope": {
        "sheets":
            SELECTED_SHEETS,

        "geographies":
            SELECTED_GEOGRAPHIES,

        "years":
            SELECTED_YEARS,

        "expected_records":
            EXPECTED_REFERENCE_RECORD_COUNT
    },

    "fixed_extraction_scope":
        (
            "Seventy-five predefined unemployment-rate "
            "observations obtained from five total-sex "
            "country/region-of-birth worksheets, five "
            "geographies and three reporting years."
        ),

    "excluded_from_reference_scope": [
        (
            "Observations from Sheets 6–15 representing "
            "sex-specific variants outside the fixed task"
        ),
        (
            "Geographies other than the five predefined "
            "reference geographies"
        ),
        (
            "Reporting years other than 2020, 2022 and 2024"
        ),
        (
            "Values requiring calculation, interpolation "
            "or external interpretation"
        )
    ],

    "representation_diagnostics":
        REPRESENTATION_DIAGNOSTICS,

}


print(
    json.dumps(
        DOCUMENT_CHARACTERISATION,
        ensure_ascii=False,
        indent=2
    )
)

{
  "document_id": "D13",
  "document_name": "Eurostat — Unemployment rates by country of birth",
  "source_file": "D13 - Unemployment rates by country of birth_2026.xlsx",
  "source_file_sha256": "13f6d031c0e5c888d5632ad20853d16e603f462cdf3b58c0dd40057824d10f9a",
  "source_format": "XLSX",
  "document_class": "Multi-sheet statistical workbook",
  "worksheet_count": 16,
  "summary_sheet": "Summary",
  "data_sheet_count": 15,
  "machine_readable": true,
  "text_extractable": true,
  "ocr_required": false,
  "time_frequency": "Annual",
  "unit_of_measure": "Percentage",
  "age_class": "From 15 to 74 years",
  "temporal_coverage": {
    "minimum_year": 2015,
    "maximum_year": 2024
  },
  "multi_sheet_structure": true,
  "repeated_data_sheet_template": true,
  "years_stored_in_columns": true,
  "value_flag_column_pairs": true,
  "merged_year_headers_present": true,
  "sheet_level_dimensions": [
    "Sex",
    "Age Class",
    "Country/Region of Birth"
  ],
  "row_level_dimensions": [
   

In [63]:
# ============================================================
# 13. Fixed extraction task
# ============================================================

EXTRACTION_TASK = """
Extract the fixed set of unemployment-rate observations represented
in the supplied D13 Eurostat workbook.

Use only the supplied workbook as evidence.

The fixed extraction scope contains:

- Sheet 1
- Sheet 2
- Sheet 3
- Sheet 4
- Sheet 5

For each of these worksheets, extract observations for:

Geographies:
- European Union - 27 countries (from 2020)
- Belgium
- Germany
- Spain
- Portugal

Reporting years:
- 2020
- 2022
- 2024

Return exactly 75 records.

For every record return:

- Category
- Topic
- Description
- Value
- Unit
- Reporting Period
- Source Location

Rules:

- Extract only information explicitly represented in the workbook.
- Preserve the association among worksheet dimensions, geography,
  reporting year, unemployment-rate value and statistical flag.
- Preserve the country/region-of-birth category represented by the
  worksheet.
- Preserve Sex and Age Class as represented in the source.
- Preserve any adjacent Eurostat statistical flag in Description.
- Use "percent" as Unit because Percentage is explicitly represented
  as the workbook's unit of measure.
- Do not calculate, interpolate, aggregate, infer, round or correct
  source values.
- Do not remove statistical flags.
- Do not replace flagged values with unflagged values.
- Use the source worksheet and value-cell reference as Source Location.
- Return exactly 75 records.
- Return valid JSON using the exact field names defined in the
  extraction schema.
- Do not include explanations before or after the JSON.
"""


print(
    EXTRACTION_TASK
)


Extract the fixed set of unemployment-rate observations represented
in the supplied D13 Eurostat workbook.

Use only the supplied workbook as evidence.

The fixed extraction scope contains:

- Sheet 1
- Sheet 2
- Sheet 3
- Sheet 4
- Sheet 5

For each of these worksheets, extract observations for:

Geographies:
- European Union - 27 countries (from 2020)
- Belgium
- Germany
- Spain
- Portugal

Reporting years:
- 2020
- 2022
- 2024

Return exactly 75 records.

For every record return:

- Category
- Topic
- Description
- Value
- Unit
- Reporting Period
- Source Location

Rules:

- Extract only information explicitly represented in the workbook.
- Preserve the association among worksheet dimensions, geography,
  reporting year, unemployment-rate value and statistical flag.
- Preserve the country/region-of-birth category represented by the
  worksheet.
- Preserve Sex and Age Class as represented in the source.
- Preserve any adjacent Eurostat statistical flag in Description.
- Use "percent

In [64]:
# ============================================================
# 14. Extraction schema
# ============================================================

EXTRACTION_SCHEMA = {
    "document_id":
        DOCUMENT_ID,

    "record_level":
        "selected_unemployment_rate_observation",

    "expected_record_count":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "fields": {
        "Category": {
            "type": [
                "string",
                "null"
            ]
        },

        "Topic": {
            "type": [
                "string",
                "null"
            ]
        },

        "Description": {
            "type": [
                "string",
                "null"
            ]
        },

        "Value": {
            "type": [
                "number",
                "null"
            ]
        },

        "Unit": {
            "type": [
                "string",
                "null"
            ],
            "expected_value":
                "percent"
        },

        "Reporting Period": {
            "type": [
                "string",
                "null"
            ]
        },

        "Source Location": {
            "type": [
                "string",
                "null"
            ]
        }
    },

    "expected_output_structure": {
        "document_id":
            DOCUMENT_ID,

        "records": [
            {
                "Category":
                    "string or null",

                "Topic":
                    "string or null",

                "Description":
                    "string or null",

                "Value":
                    "number or null",

                "Unit":
                    "string or null",

                "Reporting Period":
                    "string or null",

                "Source Location":
                    "string or null"
            }
        ]
    }
}


print(
    json.dumps(
        EXTRACTION_SCHEMA,
        ensure_ascii=False,
        indent=2
    )
)

{
  "document_id": "D13",
  "record_level": "selected_unemployment_rate_observation",
  "expected_record_count": 75,
  "fields": {
    "Category": {
      "type": [
        "string",
        "null"
      ]
    },
    "Topic": {
      "type": [
        "string",
        "null"
      ]
    },
    "Description": {
      "type": [
        "string",
        "null"
      ]
    },
    "Value": {
      "type": [
        "number",
        "null"
      ]
    },
    "Unit": {
      "type": [
        "string",
        "null"
      ],
      "expected_value": "percent"
    },
    "Reporting Period": {
      "type": [
        "string",
        "null"
      ]
    },
    "Source Location": {
      "type": [
        "string",
        "null"
      ]
    }
  },
  "expected_output_structure": {
    "document_id": "D13",
    "records": [
      {
        "Category": "string or null",
        "Topic": "string or null",
        "Description": "string or null",
        "Value": "number or null",
        "Unit":

In [65]:
# ============================================================
# 15. Reference schema
# ============================================================

REFERENCE_SCHEMA = {
    "document_id":
        DOCUMENT_ID,

    "record_level":
        "selected_unemployment_rate_observation",

    "expected_record_count":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "fields": {
        "Category":
            "Fixed reference-record category.",

        "Topic":
            (
                "Source-grounded labour-market indicator "
                "represented by the observation."
            ),

        "Description":
            (
                "Source-grounded combination of Geography, Sex, "
                "Age Class, Country/Region of Birth and, where "
                "present, the associated statistical flag."
            ),

        "Value":
            (
                "Explicit unemployment-rate value represented "
                "in the source workbook."
            ),

        "Unit":
            (
                "Percentage unit explicitly represented in "
                "the workbook."
            ),

        "Reporting Period":
            (
                "Calendar year explicitly associated with "
                "the observation."
            ),

        "Source Location":
            (
                "Source worksheet and physical value-cell "
                "coordinate."
            )
    },

    "selection_policy": {
        "selected_sheets":
            SELECTED_SHEETS,

        "selected_geographies":
            SELECTED_GEOGRAPHIES,

        "selected_years":
            SELECTED_YEARS
    },

    "preservation_rules": [
        "Preserve exact source values.",
        "Preserve worksheet-level dimensions.",
        "Preserve geography-to-value association.",
        "Preserve year-to-value association.",
        "Preserve statistical flags when represented.",
        "Do not calculate or derive values.",
        "Do not interpolate observations.",
        "Do not aggregate source observations.",
        "Do not repair or replace flagged values."
    ],

    "construction_method":
        (
            "Deterministic selection of predefined workbook "
            "observations followed by programmatic integrity "
            "verification."
        ),

    "branch_reuse":
        (
            "The same fixed reference dataset is reused for "
            "Branches A, B and C."
        )
}


print(
    json.dumps(
        REFERENCE_SCHEMA,
        ensure_ascii=False,
        indent=2
    )
)

{
  "document_id": "D13",
  "record_level": "selected_unemployment_rate_observation",
  "expected_record_count": 75,
  "fields": {
    "Category": "Fixed reference-record category.",
    "Topic": "Source-grounded labour-market indicator represented by the observation.",
    "Description": "Source-grounded combination of Geography, Sex, Age Class, Country/Region of Birth and, where present, the associated statistical flag.",
    "Value": "Explicit unemployment-rate value represented in the source workbook.",
    "Unit": "Percentage unit explicitly represented in the workbook.",
    "Reporting Period": "Calendar year explicitly associated with the observation.",
    "Source Location": "Source worksheet and physical value-cell coordinate."
  },
  "selection_policy": {
    "selected_sheets": [
      "Sheet 1",
      "Sheet 2",
      "Sheet 3",
      "Sheet 4",
      "Sheet 5"
    ],
    "selected_geographies": [
      "European Union - 27 countries (from 2020)",
      "Belgium",
      "Ger

In [66]:
# ============================================================
# 16. Indicator-level document assessment
# ============================================================

indicator_assessment = [
    {
        "Dimension":
            "Structural Readiness",

        "Indicator":
            "Reading Order Quality",

        "Score":
            "Medium",

        "Evidence Source":
            "Workbook profiling + manual worksheet inspection",

        "Justification":
            (
                "The workbook is machine-readable, but a complete "
                "observation must be reconstructed from worksheet-level "
                "dimensions, a geography row, a reporting-year column "
                "and its adjacent statistical-flag cell."
            )
    },

    {
        "Dimension":
            "Structural Readiness",

        "Indicator":
            "Table Structure Integrity",

        "Score":
            "Medium",

        "Evidence Source":
            "Workbook structure and merged-header inspection",

        "Justification":
            (
                "Data sheets follow a regular repeated table template, "
                "but annual headings span value/flag column pairs and "
                "correct extraction requires preserving the association "
                "between each value and its adjacent statistical flag."
            )
    },

    {
        "Dimension":
            "Structural Readiness",

        "Indicator":
            "Section/Header Hierarchy",

        "Score":
            "Medium",

        "Evidence Source":
            "Workbook and worksheet hierarchy inspection",

        "Justification":
            (
                "Interpretation depends on several hierarchy levels: "
                "worksheet-level dimensions, metadata rows, year headers, "
                "geography rows and adjacent observation-flag columns."
            )
    },

    {
        "Dimension":
            "Visual/OCR Readiness",

        "Indicator":
            "Sharpness",

        "Score":
            "Low",

        "Evidence Source":
            "File-format inspection",

        "Justification":
            (
                "The workbook contains natively machine-readable cells, "
                "so image sharpness does not constrain information "
                "recovery."
            )
    },

    {
        "Dimension":
            "Visual/OCR Readiness",

        "Indicator":
            "Noise / Degradation",

        "Score":
            "Low",

        "Evidence Source":
            "File-format inspection",

        "Justification":
            (
                "No scanning noise, blur or visual degradation affects "
                "the machine-readable spreadsheet content."
            )
    },

    {
        "Dimension":
            "Visual/OCR Readiness",

        "Indicator":
            "OCR Dependency",

        "Score":
            "Low",

        "Evidence Source":
            "Automated workbook inspection",

        "Justification":
            (
                "All relevant metadata, labels, values and statistical "
                "flags are stored directly in spreadsheet cells and OCR "
                "is not required."
            )
    },

    {
        "Dimension":
            "Semantic Quality",

        "Indicator":
            "Terminology Consistency",

        "Score":
            "Low",

        "Evidence Source":
            "Summary metadata + worksheet inspection",

        "Justification":
            (
                "Eurostat dimension labels, geography labels, country/"
                "region-of-birth categories and measurement terminology "
                "are applied consistently across the repeated worksheets."
            )
    },

    {
        "Dimension":
            "Semantic Quality",

        "Indicator":
            "Schema Alignment",

        "Score":
            "Medium",

        "Evidence Source":
            "Reference-schema comparison",

        "Justification":
            (
                "All required concepts are represented in the workbook, "
                "but they are distributed across worksheet metadata, "
                "geography rows, year columns and adjacent flag cells "
                "rather than stored as one complete row-wise observation."
            )
    },

    {
        "Dimension":
            "Semantic Quality",

        "Indicator":
            "Numerical Density",

        "Score":
            "High",

        "Evidence Source":
            "Automated workbook profiling + manual inspection",

        "Justification":
            (
                "The data worksheets contain dense annual statistical "
                "observations, with unemployment-rate values repeated "
                "across many geographies, years and dimensional variants."
            )
    },

    {
        "Dimension":
            "Completeness and Consistency",

        "Indicator":
            "Required Field Presence",

        "Score":
            "Low",

        "Evidence Source":
            "Reference-value verification",

        "Justification":
            (
                "All required dimensions and numerical values are present "
                "for the predefined 75-record extraction scope."
            )
    },

    {
        "Dimension":
            "Completeness and Consistency",

        "Indicator":
            "Internal Consistency",

        "Score":
            "Low",

        "Evidence Source":
            "Workbook-template and reference validation",

        "Justification":
            (
                "The fifteen data sheets follow a consistent Eurostat "
                "template and the selected observations use the same "
                "dimensional and value/flag conventions."
            )
    },

    {
        "Dimension":
            "Representation and Normalisation Complexity",

        "Indicator":
            "Format Heterogeneity",

        "Score":
            "Medium",

        "Evidence Source":
            "Workbook profiling",

        "Justification":
            (
                "The source uses one file format but distributes "
                "information across a Summary worksheet and fifteen "
                "data worksheets whose dimensions vary by sex and "
                "country/region-of-birth category."
            )
    },

    {
        "Dimension":
            "Representation and Normalisation Complexity",

        "Indicator":
            "Unit / Label Variability",

        "Score":
            "Medium",

        "Evidence Source":
            "Summary metadata + source representation inspection",

        "Justification":
            (
                "The measurement unit is consistent, but correct "
                "representation requires preserving multiple geography "
                "labels, birth categories, sex categories, years, "
                "sheet-level labels and statistical flags."
            )
    }
]


indicator_assessment_df = pd.DataFrame(
    indicator_assessment
)

display(
    indicator_assessment_df
)

,Dimension,Indicator,Score,Evidence Source,Justification
0,Structural Readiness,Reading Order Quality,Medium,Workbook profiling + manual worksheet inspection,"The workbook is machine-readable, but a comple..."
1,Structural Readiness,Table Structure Integrity,Medium,Workbook structure and merged-header inspection,Data sheets follow a regular repeated table te...
2,Structural Readiness,Section/Header Hierarchy,Medium,Workbook and worksheet hierarchy inspection,Interpretation depends on several hierarchy le...
3,Visual/OCR Readiness,Sharpness,Low,File-format inspection,The workbook contains natively machine-readabl...
4,Visual/OCR Readiness,Noise / Degradation,Low,File-format inspection,"No scanning noise, blur or visual degradation ..."
5,Visual/OCR Readiness,OCR Dependency,Low,Automated workbook inspection,"All relevant metadata, labels, values and stat..."
6,Semantic Quality,Terminology Consistency,Low,Summary metadata + worksheet inspection,"Eurostat dimension labels, geography labels, c..."
7,Semantic Quality,Schema Alignment,Medium,Reference-schema comparison,All required concepts are represented in the w...
8,Semantic Quality,Numerical Density,High,Automated workbook profiling + manual inspection,The data worksheets contain dense annual stati...
9,Completeness and Consistency,Required Field Presence,Low,Reference-value verification,All required dimensions and numerical values a...


In [67]:
# ============================================================
# 17. Validate indicator assessment
# ============================================================

VALID_SCORES = {
    "Low",
    "Medium",
    "High"
}


expected_indicators = {
    "Reading Order Quality",
    "Table Structure Integrity",
    "Section/Header Hierarchy",
    "Sharpness",
    "Noise / Degradation",
    "OCR Dependency",
    "Terminology Consistency",
    "Schema Alignment",
    "Numerical Density",
    "Required Field Presence",
    "Internal Consistency",
    "Format Heterogeneity",
    "Unit / Label Variability"
}


invalid_scores = (
    set(
        indicator_assessment_df[
            "Score"
        ].dropna().unique()
    )
    - VALID_SCORES
)


observed_indicators = set(
    indicator_assessment_df[
        "Indicator"
    ]
)


missing_indicators = (
    expected_indicators
    - observed_indicators
)


unexpected_indicators = (
    observed_indicators
    - expected_indicators
)


if invalid_scores:
    raise ValueError(
        f"Invalid indicator scores: {invalid_scores}"
    )


if missing_indicators:
    raise ValueError(
        f"Missing indicators: {missing_indicators}"
    )


if unexpected_indicators:
    raise ValueError(
        f"Unexpected indicators: {unexpected_indicators}"
    )


if len(
    indicator_assessment_df
) != len(
    expected_indicators
):
    raise ValueError(
        "Duplicate indicator rows detected."
    )


print(
    "Indicator assessment validation passed."
)

Indicator assessment validation passed.


In [68]:
# ============================================================
# 18. Dimension-level assessment
# ============================================================

score_to_numeric = {
    "Low": 1,
    "Medium": 2,
    "High": 3
}


indicator_assessment_df[
    "Numeric Score"
] = indicator_assessment_df[
    "Score"
].map(
    score_to_numeric
)


dimension_assessment_df = (
    indicator_assessment_df
    .groupby(
        "Dimension",
        as_index=False
    )
    .agg(
        Mean_Score=(
            "Numeric Score",
            "mean"
        ),

        Number_of_Indicators=(
            "Indicator",
            "count"
        )
    )
)


def classify_dimension_score(
    mean_score
):
    if mean_score < 1.5:
        return "Low"

    elif mean_score < 2.5:
        return "Medium"

    else:
        return "High"


dimension_assessment_df[
    "Dimension Score"
] = dimension_assessment_df[
    "Mean_Score"
].apply(
    classify_dimension_score
)


dimension_assessment_df[
    "Mean_Score"
] = dimension_assessment_df[
    "Mean_Score"
].round(
    2
)


display(
    dimension_assessment_df
)

,Dimension,Mean_Score,Number_of_Indicators,Dimension Score
0,Completeness and Consistency,1.0,2,Low
1,Representation and Normalisation Complexity,2.0,2,Medium
2,Semantic Quality,2.0,3,Medium
3,Structural Readiness,2.0,3,Medium
4,Visual/OCR Readiness,1.0,3,Low


In [69]:
# ============================================================
# 19. Structured quality-assessment evidence
# ============================================================

QUALITY_EVIDENCE = {
    "document_id":
        DOCUMENT_ID,

    "assessment_basis":
        (
            "Observed document evidence was mapped to the "
            "predefined Low, Medium, and High operational "
            "criteria defined in Table 3.3 of the methodology."
        ),

    "evidence_method":
        (
            "Evidence was obtained through automated profiling "
            "where measurable characteristics could be derived "
            "programmatically and through documented manual "
            "inspection where qualitative assessment was required."
        ),

    "indicators":
        indicator_assessment_df[
            [
                "Dimension",
                "Indicator",
                "Score",
                "Evidence Source",
                "Justification"
            ]
        ].to_dict(
            orient="records"
        ),

    "dimension_aggregation": {
        "encoding": {
            "Low": 1,
            "Medium": 2,
            "High": 3
        },

        "aggregation":
            (
                "Arithmetic mean of indicator scores "
                "within each dimension."
            ),

        "classification_rule": {
            "Low":
                "mean < 1.5",

            "Medium":
                "1.5 <= mean < 2.5",

            "High":
                "mean >= 2.5"
        }
    },

    "dimensions":
        dimension_assessment_df[
            [
                "Dimension",
                "Mean_Score",
                "Dimension Score"
            ]
        ].to_dict(
            orient="records"
        )
}

In [70]:
# ============================================================
# 20. Reference metadata and integrity summary
# ============================================================

REFERENCE_SUMMARY = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "expected_reference_records":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "observed_reference_records":
        int(
            len(reference_values_df)
        ),

    "reference_record_count_valid":
        bool(
            reference_record_count_valid
        ),

    "selected_sheet_count":
        len(
            SELECTED_SHEETS
        ),

    "selected_geography_count":
        len(
            SELECTED_GEOGRAPHIES
        ),

    "selected_year_count":
        len(
            SELECTED_YEARS
        ),

    "selected_sheets":
        SELECTED_SHEETS,

    "selected_geographies":
        SELECTED_GEOGRAPHIES,

    "selected_years":
        SELECTED_YEARS,

    "flagged_reference_records":
        flagged_reference_records,

    "unflagged_reference_records":
        unflagged_reference_records,

    "statistical_flag_counts":
        statistical_flag_counts,

    "reference_values_numeric":
        bool(
            reference_values_numeric
        ),

    "reference_periods_valid":
        bool(
            reference_periods_valid
        ),

    "reference_locations_valid":
        bool(
            reference_locations_valid
        ),

    "reference_unit":
        "percent",

    "reference_integrity_valid":
        bool(
            reference_integrity_valid
        ),

    "reference_values_branch_independent":
        True,

    "reference_values_reused_across_branches":
        True
}


REFERENCE_INTEGRITY = {
    "document_id":
        DOCUMENT_ID,

    "workbook_structure_valid":
        bool(
            workbook_structure_valid
        ),

    "summary_metadata_valid":
        bool(
            summary_metadata_valid
        ),

    "selected_observations_valid":
        bool(
            selected_observations_valid
        ),

    "reference_schema_valid":
        bool(
            reference_schema_valid
        ),

    "reference_record_count_valid":
        bool(
            reference_record_count_valid
        ),

    "reference_null_count":
        int(
            reference_null_count
        ),

    "reference_duplicate_count":
        int(
            reference_duplicate_count
        ),

    "reference_values_numeric":
        bool(
            reference_values_numeric
        ),

    "reference_periods_valid":
        bool(
            reference_periods_valid
        ),

    "reference_locations_valid":
        bool(
            reference_locations_valid
        ),

    "reference_category_valid":
        bool(
            reference_category_valid
        ),

    "reference_topic_valid":
        bool(
            reference_topic_valid
        ),

    "reference_unit_valid":
        bool(
            reference_unit_valid
        ),

    "reference_integrity_passed":
        bool(
            reference_integrity_valid
        )
}


REFERENCE_METADATA = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "source_file":
        SOURCE_PATH.name,

    "source_file_sha256":
        SOURCE_SHA256,

    "source_format":
        SOURCE_FORMAT,

    "reference_file":
        REFERENCE_VALUES_PATH.name,

    "reference_construction_method":
        (
            "Deterministic selection of predefined "
            "source observations followed by "
            "programmatic integrity verification."
        ),

    "expected_reference_record_count":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "observed_reference_record_count":
        int(
            len(reference_values_df)
        ),

    "reference_fields":
        REFERENCE_FIELDS,

    "selected_sheets":
        SELECTED_SHEETS,

    "selected_geographies":
        SELECTED_GEOGRAPHIES,

    "selected_years":
        SELECTED_YEARS,

    "selection_policy":
        (
            "Five total-sex worksheets representing distinct "
            "country/region-of-birth categories, five geographies "
            "and three predefined reporting years."
        ),

    "selection_reason":
        (
            "The fixed 75-record subset preserves variation across "
            "country-of-birth categories, geographies, reporting "
            "years and statistical flags while remaining manageable "
            "for repeated extraction across all branches."
        ),

    "unit_source":
        (
            "Percentage is explicitly represented as the "
            "unit of measure in the workbook."
        ),

    "raw_source_values_modified":
        False,

    "values_calculated":
        False,

    "values_aggregated":
        False,

    "values_interpolated":
        False,

    "values_rounded":
        False,

    "manual_correction_applied":
        False,

    "semantic_inference_applied":
        False,

    "statistical_flags_preserved":
        True,

    "statistical_flags_preserved_in_description":
        True,

    "reference_values_branch_independent":
        True,

    "reference_values_to_be_reused_for_branches": [
        "A",
        "B",
        "C"
    ],

    "notes":
        (
            "D13 is a multi-sheet Eurostat workbook in which "
            "observation dimensions are distributed across worksheet "
            "metadata, geography rows, year columns and adjacent "
            "statistical-flag cells. The fixed reference dataset "
            "preserves these associations without modifying values."
        )
}


print(
    json.dumps(
        REFERENCE_SUMMARY,
        ensure_ascii=False,
        indent=2
    )
)

print(
    json.dumps(
        REFERENCE_INTEGRITY,
        ensure_ascii=False,
        indent=2
    )
)

{
  "document_id": "D13",
  "document_name": "Eurostat — Unemployment rates by country of birth",
  "expected_reference_records": 75,
  "observed_reference_records": 75,
  "reference_record_count_valid": true,
  "selected_sheet_count": 5,
  "selected_geography_count": 5,
  "selected_year_count": 3,
  "selected_sheets": [
    "Sheet 1",
    "Sheet 2",
    "Sheet 3",
    "Sheet 4",
    "Sheet 5"
  ],
  "selected_geographies": [
    "European Union - 27 countries (from 2020)",
    "Belgium",
    "Germany",
    "Spain",
    "Portugal"
  ],
  "selected_years": [
    2020,
    2022,
    2024
  ],
  "flagged_reference_records": 17,
  "unflagged_reference_records": 58,
  "statistical_flag_counts": {
    "No flag": 58,
    "d": 10,
    "b": 5,
    "u": 2
  },
  "reference_values_numeric": true,
  "reference_periods_valid": true,
  "reference_locations_valid": true,
  "reference_unit": "percent",
  "reference_integrity_valid": true,
  "reference_values_branch_independent": true,
  "reference_val

In [71]:
# ============================================================
# 21. Export Stage 1 artefacts
# ============================================================

sheet_characterisation_df.to_csv(
    SHEET_CHARACTERISATION_PATH,
    index=False,
    encoding="utf-8-sig"
)

selection_matrix_df.to_csv(
    SELECTION_MATRIX_PATH,
    index=False,
    encoding="utf-8-sig"
)

reference_values_df.to_csv(
    REFERENCE_VALUES_PATH,
    index=False,
    encoding="utf-8-sig"
)


REFERENCE_VALUES_JSON_PATH.write_text(
    json.dumps(
        reference_values_df.to_dict(
            orient="records"
        ),
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)


indicator_assessment_df[
    [
        "Dimension",
        "Indicator",
        "Score",
        "Evidence Source",
        "Justification"
    ]
].to_csv(
    INDICATOR_ASSESSMENT_PATH,
    index=False,
    encoding="utf-8-sig"
)


dimension_assessment_df.to_csv(
    DIMENSION_ASSESSMENT_PATH,
    index=False,
    encoding="utf-8-sig"
)


json_outputs = [
    (
        DOCUMENT_CHARACTERISATION_PATH,
        DOCUMENT_CHARACTERISATION
    ),
    (
        WORKBOOK_INTEGRITY_PATH,
        WORKBOOK_INTEGRITY
    ),
    (
        REFERENCE_SCHEMA_PATH,
        REFERENCE_SCHEMA
    ),
    (
        EXTRACTION_SCHEMA_PATH,
        EXTRACTION_SCHEMA
    ),
    (
        REFERENCE_SUMMARY_PATH,
        REFERENCE_SUMMARY
    ),
    (
        REFERENCE_INTEGRITY_PATH,
        REFERENCE_INTEGRITY
    ),
    (
        REFERENCE_METADATA_PATH,
        REFERENCE_METADATA
    ),
    (
        QUALITY_EVIDENCE_PATH,
        QUALITY_EVIDENCE
    )
]


for output_path, content in json_outputs:

    output_path.write_text(
        json.dumps(
            content,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8",
        newline="\n"
    )


EXTRACTION_TASK_PATH.write_text(
    EXTRACTION_TASK.strip(),
    encoding="utf-8",
    newline="\n"
)


print(
    "D13 Stage 1 outputs exported successfully."
)

D13 Stage 1 outputs exported successfully.


In [72]:
# ============================================================
# 22. Final Stage 1 checks
# ============================================================

GENERATED_OUTPUTS = [
    DOCUMENT_CHARACTERISATION_PATH,
    WORKBOOK_INTEGRITY_PATH,
    SHEET_CHARACTERISATION_PATH,
    SELECTION_MATRIX_PATH,
    REFERENCE_VALUES_PATH,
    REFERENCE_VALUES_JSON_PATH,
    REFERENCE_SCHEMA_PATH,
    REFERENCE_SUMMARY_PATH,
    REFERENCE_INTEGRITY_PATH,
    REFERENCE_METADATA_PATH,
    EXTRACTION_SCHEMA_PATH,
    EXTRACTION_TASK_PATH,
    INDICATOR_ASSESSMENT_PATH,
    DIMENSION_ASSESSMENT_PATH,
    QUALITY_EVIDENCE_PATH
]


missing_output_files = [
    path.name
    for path in GENERATED_OUTPUTS
    if not path.exists()
]


if missing_output_files:
    raise AssertionError(
        f"Missing D13 outputs: "
        f"{missing_output_files}"
    )


ALL_STAGE_1_CHECKS_PASSED = all(
    [
        workbook_structure_valid,
        summary_metadata_valid,
        selected_observations_valid,
        reference_integrity_valid,
        len(indicator_assessment_df)
        == 13,
        len(dimension_assessment_df)
        == 5,
        not missing_output_files
    ]
)


FINAL_SUMMARY = {
    "document_id":
        DOCUMENT_ID,

    "workbook_structure_valid":
        bool(
            workbook_structure_valid
        ),

    "summary_metadata_valid":
        bool(
            summary_metadata_valid
        ),

    "selected_observations_valid":
        bool(
            selected_observations_valid
        ),

    "reference_integrity_valid":
        bool(
            reference_integrity_valid
        ),

    "worksheet_count":
        len(
            sheet_names
        ),

    "selected_sheet_count":
        len(
            SELECTED_SHEETS
        ),

    "selected_geography_count":
        len(
            SELECTED_GEOGRAPHIES
        ),

    "selected_year_count":
        len(
            SELECTED_YEARS
        ),

    "reference_records":
        int(
            len(reference_values_df)
        ),

    "flagged_reference_records":
        flagged_reference_records,

    "indicator_count":
        int(
            len(indicator_assessment_df)
        ),

    "dimension_count":
        int(
            len(dimension_assessment_df)
        ),

    "all_outputs_exist":
        bool(
            not missing_output_files
        ),

    "all_stage_1_checks_passed":
        bool(
            ALL_STAGE_1_CHECKS_PASSED
        ),

    "outputs_created": [
        path.name
        for path in GENERATED_OUTPUTS
    ]
}


print(
    json.dumps(
        FINAL_SUMMARY,
        ensure_ascii=False,
        indent=2
    )
)


if not ALL_STAGE_1_CHECKS_PASSED:

    raise AssertionError(
        "D13 Stage 1 preparation failed."
    )


print(
    "\nD13 Stage 1 completed successfully."
)

{
  "document_id": "D13",
  "workbook_structure_valid": true,
  "summary_metadata_valid": true,
  "selected_observations_valid": true,
  "reference_integrity_valid": true,
  "worksheet_count": 16,
  "selected_sheet_count": 5,
  "selected_geography_count": 5,
  "selected_year_count": 3,
  "reference_records": 75,
  "flagged_reference_records": 17,
  "indicator_count": 13,
  "dimension_count": 5,
  "all_outputs_exist": true,
  "all_stage_1_checks_passed": true,
  "outputs_created": [
    "D13_document_characterisation.json",
    "D13_workbook_integrity.json",
    "D13_sheet_characterisation.csv",
    "D13_selection_matrix.csv",
    "D13_reference_values.csv",
    "D13_reference_values.json",
    "D13_reference_schema.json",
    "D13_reference_summary.json",
    "D13_reference_integrity.json",
    "D13_reference_metadata.json",
    "D13_extraction_schema.json",
    "D13_extraction_task.txt",
    "D13_indicator_assessment.csv",
    "D13_dimension_assessment.csv",
    "D13_quality_evidence.

In [73]:
# ============================================================
# 23. List generated outputs
# ============================================================

print(
    "Generated D13 Stage 1 files:\n"
)


for path in GENERATED_OUTPUTS:

    print(
        "-",
        path.name,
        "| exists:",
        path.exists()
    )

Generated D13 Stage 1 files:

- D13_document_characterisation.json | exists: True
- D13_workbook_integrity.json | exists: True
- D13_sheet_characterisation.csv | exists: True
- D13_selection_matrix.csv | exists: True
- D13_reference_values.csv | exists: True
- D13_reference_values.json | exists: True
- D13_reference_schema.json | exists: True
- D13_reference_summary.json | exists: True
- D13_reference_integrity.json | exists: True
- D13_reference_metadata.json | exists: True
- D13_extraction_schema.json | exists: True
- D13_extraction_task.txt | exists: True
- D13_indicator_assessment.csv | exists: True
- D13_dimension_assessment.csv | exists: True
- D13_quality_evidence.json | exists: True
